# 10 Operating points: precision targets and the two-control policy

Sensitivity study of the M9 decision policy. The locked policy has one control c = 0.7 and a review band symmetric around 0.5. This notebook splits it into two controls, a correction threshold that governs precision and recall and a keep threshold that governs the review load, asks what threshold delivers a minimum precision on unseen stations, and shows the cost of each target in recall, Energy IoU and remaining false corrections. The headline result at c = 0.7 is not changed.

Abbreviations used here: **RPF** is reverse power flow, the condition where a distribution substation exports power because rooftop solar exceeds local demand; a *wrong RPF sign* is a meter recording that stores the export as an import. **M7** is the deterministic threshold rule, **M8** the two-stage XGBoost classifier and **M9** the compact counterfactual method (revision 2). **MW** and **MWh** are megawatts and megawatt-hours; one interval is 15 minutes.

**Inputs.** `06_site_days/site_days.parquet` (M9 rows: held-out probability, candidate energies, labels) and `01_folds/fold_manifest.csv` (calibration stations per fold). M8 is not needed.

**Outputs.** `outputs/01_final_evaluation/10_operating_points/`: `frontier.csv`, `heatmap.csv`, `targets.csv`, `selections.csv`, `bootstrap.csv`, `stations.csv` and six figures per cohort (A to F); `manifests/10_operating_points.json`.

**Approximate runtime.** About two minutes (station bootstrap per target and rule).

**Prerequisites.** Notebooks 01, 05 and 06.

**Main process.**

1. Frontier: every metric against the correction threshold with the symmetric keep rule.
2. Threshold selection per fold on the calibration stations only, by rule L (expected precision from the calibrated probability, label-free) and rule E (observed precision on the calibration stations' labels); applied to the held-out station; pooled.
3. Grid over both thresholds for the review side; per-station consistency at chosen points; station bootstrap of achieved precision.
4. Figures: A frontier, B target against achieved, C cost of each target, D review heat maps, E per station, F the Ausgrid one-pager. A and C are the paper figures; the rest are documentation and Ausgrid material.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def article_root() -> Path:
    """Locate publication/2_journal_article from JupyterLab, VS Code or the repository root."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "final_eval" / "cli.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article"
        if (nested / "final_eval" / "cli.py").exists():
            return nested
    raise FileNotFoundError("Could not locate publication/2_journal_article.")


ARTICLE = article_root()
sys.path.insert(0, str(ARTICLE))
sys.path.insert(0, str(ARTICLE.parents[1] / "src"))  # the repository's pynrpf package

from final_eval import cli, config  # noqa: E402

SETTINGS = config.load()  # verifies the frozen dataset hashes
OUT = SETTINGS.output_root()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Article root:", ARTICLE.relative_to(ARTICLE.parents[2]))

## 2. Leakage rule

A threshold chosen on the held-out predictions and then scored on them would be tuning on the test set. Every threshold here is chosen on the fold's calibration stations (the other stations of the same cohort, headline-confidence days) and applied once to the held-out station; the tests in `tests/test_final_eval_operating_points.py` check that flipping the held-out labels leaves the thresholds unchanged.

In [ ]:
display(pd.Series(SETTINGS["operating_points"]["targets"], name="precision_targets"))

## 3. Run the study

In [ ]:
result = cli.stage_operating_points(SETTINGS)

## 4. Target table

One row per target, what it applies to (energy precision, site-day precision, or both) and rule. `c_correct_mean` is the mean threshold across folds; `binding` says which precision set it. Achieved values are pooled held-out results.

In [ ]:
targets = result["targets"]
beta = targets[targets["cohort"] == "beta"]
display(beta[["target", "applies", "rule", "c_correct_mean", "c_correct_min", "c_correct_max", "binding", "energy_precision",
              "day_precision", "day_recall", "energy_iou", "fp_per_station_year", "review_days_per_station_year", "all_attainable"]].round(3))

## 5. Figures, Beta 'sure'

A and C are the paper figures.

In [ ]:
for name in ["figA_precision_recall_frontier_beta", "figC_cost_of_precision_target_beta", "figB_target_vs_achieved_beta",
             "figD_review_side_heatmaps_beta", "figE_per_station_operating_points_beta", "figF_ausgrid_one_pager_beta"]:
    display(Image(filename=OUT / "10_operating_points" / f"{name}.png"))

## 6. Alpha appendix

The same study on Alpha, read with its label-contiguity ceiling in mind (energy precision cannot exceed about 0.904 there).

In [ ]:
for name in ["figA_precision_recall_frontier_alpha", "figB_target_vs_achieved_alpha", "figC_cost_of_precision_target_alpha"]:
    display(Image(filename=OUT / "10_operating_points" / f"{name}.png"))

## Conclusion

The study reports a procedure (choose the threshold from a precision target on labelled or unlabelled stations) and the trade-off it implies; it does not move the headline operating point. Which target Ausgrid adopts is a decision to make with Fig C and Fig F in front of them.